# Critic v2: learn taste from people, and know when nothing is good enough

**Where v1 stopped.** Critic v1 learned one thing: a design beats a broken copy of
itself. It never saw two *good* designs compared, so among valid designs its order
is close to noise. That is the part a designer cares about.

**What this notebook builds, from the Taste Lab's answers:**

1. **Critic v2**, a ranker that learns which of four real designs people would send.
2. **The quality bar**, a second, small model that answers a different question:
   *would anyone send this at all?* A ranker only orders; its best of four bad
   designs is still bad.

**Whose taste?** Not one person's. Every rater says what they are (designer, design
student, other). Designers' answers count three times as much in training, and
**they are the answer key**: v2 is scored on their answers when there are at least
100 of them.

**The rule, written before looking at any result.** v2 replaces v1 only if, out of
fold, on the answer key:
- it puts designs in the rater's order more often than v1 in **at least 95%** of
  paired bootstrap resamples (`pairs`);
- it is **no worse** than v1 at picking the one they picked (`top1`);
- it still knows a broken design from a good one (synthetic `base_wins` ≥ 0.95);
- it learned from **at least 150** answers.

If any check fails, nothing changes: v1 stays, and the report says which check
failed. The bar has its own, smaller rule (step 8). Both models are switched on by
files the product reads (`critic/model/ACTIVE`, `critic/model/bar.json`); deleting
them switches back.

**Before you start:** export the Taste Lab (Taste Lab page → Export, or
`GET /api/lab/export`) and save it as `ml/critic/labels/taste-lab.jsonl`. That
folder is in `.gitignore`: the export holds client posts and must never be
committed. Without the file this notebook runs on **made-up answers** (a dry run)
and saves nothing.

## Step 0 — Set up

`stratum_worker.critic.taste` reads the export and does the counting: labels,
folds, marks. `stratum_worker.critic.bar` is the quality bar. LightGBM trains the
ranker, as in v1.

In [ ]:
import json
import tempfile
import time
from dataclasses import replace
from datetime import date
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
from stratum_worker.critic import bar as quality
from stratum_worker.critic import taste
from stratum_worker.critic.data import SPLITS, groups
from stratum_worker.critic.features import FEATURES
from stratum_worker.critic.ranker import (
    DEFAULT_MODEL,
    MODELS,
    Forest,
    heuristic,
    save_model,
    write_active,
)

ROOT = Path.cwd().resolve()
while not (ROOT / "templates").is_dir():
    ROOT = ROOT.parent
LABELS = ROOT / "ml" / "critic" / "labels" / "taste-lab.jsonl"
REPORT = ROOT / "ml" / "critic" / "report-v2.json"
DRY_RUN = not LABELS.is_file()

# Rater numbers of designers who rated before the page asked for a role (see the
# table in step 1). Leave empty if every designer said so on the page.
EXPERT_RATERS: set[int] = set()
# An answer quicker than this is someone clicking through, not looking at four designs.
MIN_MS = 1500
print("repository:", ROOT)
print("lightgbm", lgb.__version__, "| measurements per design:", len(FEATURES))
print("DRY RUN on made-up answers: nothing will be saved" if DRY_RUN else f"labels: {LABELS}")

## Step 1 — What is in the data

Each answer is one person, one screen of four designs for one real post. With the
newer page, they also tick every design they would send as it is.

| | best | would send | unmarked | never send |
|---|---|---|---|---|
| label (higher is better) | 3 | 2 | 1 | 0 |

An older answer (no ticks) is best 2, unmarked 1, never 0. "None of these" with no
never-send orders nothing, so the ranker skips it; the bar still learns from it
(every design on it was below the bar).

In [ ]:
if DRY_RUN:
    # Two made-up raters who share most of a taste: a designer, and someone noisier.
    rng = np.random.default_rng(0)
    TASTE = rng.normal(size=len(FEATURES)) * (rng.random(len(FEATURES)) < 0.3)
    other_taste = TASTE + rng.normal(scale=0.6, size=len(FEATURES))
    answers = taste.simulate(
        260, TASTE, noise=0.5, bar=0.0, posts=45, rater=1, role="designer", prefix="d", seed=1
    ) + taste.simulate(
        200, other_taste, noise=0.9, bar=0.3, posts=45, rater=2, role="other", prefix="o", seed=2
    )
    answers = [replace(one, ms=4000) for one in answers]
else:
    answers = taste.read_export(LABELS.read_text(encoding="utf-8").splitlines())

too_fast = [one for one in answers if 0 < one.ms < MIN_MS]
kept = [one for one in answers if not 0 < one.ms < MIN_MS]
rows = []
for rater in sorted({one.rater for one in kept}):
    mine = [one for one in kept if one.rater == rater]
    first = [one for one in mine if not one.repeat]
    rows.append(
        {
            "rater": rater,
            "role": mine[-1].role or "(not said)",
            "answers": len(first),
            "repeats": len(mine) - len(first),
            "none of these": sum(one.picked is None for one in first),
            "with a never-send": sum(bool(one.never) for one in first),
            "with would-send ticks": sum(one.send is not None for one in first),
            "median seconds": round(float(np.median([one.ms for one in first])) / 1000, 1),
        }
    )
display(pd.DataFrame(rows).set_index("rater"))

TRAIN = taste.training(kept)
KEY_ONLY = taste.experts(TRAIN, EXPERT_RATERS)
ON_KEY = len(KEY_ONLY) >= 100
KEY_NAME = "designers" if ON_KEY else "everyone (fewer than 100 designer answers)"
key_ids = {id(one) for one in (KEY_ONLY if ON_KEY else TRAIN)}
KEY_AT = [i for i, one in enumerate(TRAIN) if id(one) in key_ids]
print(f"left out as too quick: {len(too_fast)}")
posts = len({one.post for one in TRAIN})
print(f"answers that order something (training): {len(TRAIN)} on {posts} posts")
print(f"answer key: {KEY_NAME}, {len(KEY_AT)} answers")

**Read the table before anything else.** A rater with a median under two seconds,
or who almost always says "none of these", is telling you about their patience, not
their taste. Under about 150 training answers, every number below is too uncertain to
act on: rate more first.

## Step 2 — The ceiling: how much do people agree with themselves?

One answer in ten (after the tenth) was a screen the rater had already answered.
**Self-consistency** is how often they picked the same design again. **Agreement**
is how often two raters picked the same design on the same screen.

No model can be expected to agree with a person more often than that person agrees
with themselves. If self-consistency is 0.6, a `top1` of 0.5 is good.

In [ ]:
ceiling = {}
for rater in sorted({one.rater for one in kept}):
    n, rate = taste.consistency([one for one in kept if one.rater == rater])
    same = None if rate is None else round(rate, 2)
    ceiling[rater] = {"repeats": n, "picked the same again": same}
display(pd.DataFrame(ceiling).T)
pairs, rate = taste.agreement(kept)
said = f", {rate:.2f} picked the same" if pairs else ""
print(f"agreement between raters: {pairs} pairs{said}")
print(f"chance of picking the same design at random: {taste.chance(TRAIN):.2f}")

## Step 3 — Baselines: what v2 has to beat, with no training at all

- **chance**: a random order;
- **the product's ranking**: the score the Taste Lab used to choose these designs
  (critic v1 plus how well the photo sits);
- **the hand-written rule** from v1's notebook;
- **critic v1**.

The marks, each with a 95% range over answers:
- `top1`: the design scored best is the one picked;
- `pairs`: of every pair the answer puts in order, the share the scores put in the
  same order (0.5 is a coin);
- `never_last`: a never-send design is scored below the pick;
- `top_ok`: the design scored best is one they would send (answers with ticks only).

In [ ]:
V1 = Forest.load(DEFAULT_MODEL)
rng = np.random.default_rng(0)
SCORES = {
    "chance": [rng.random(len(one.features)) for one in TRAIN],
    "the product's ranking": [one.product for one in TRAIN],
    "hand-written rule": taste.score_all(heuristic, TRAIN),
    "critic v1": taste.score_all(V1.predict, TRAIN),
}


def table(names, at):
    out = {}
    for name in names:
        got = taste.with_range([SCORES[name][i] for i in at], [TRAIN[i] for i in at], repeats=500)
        out[name] = {
            key: f"{value:.2f} ({low:.2f}-{high:.2f})"
            for key, (low, value, high) in got.items()
            if np.isfinite(value)
        }
    return pd.DataFrame(out).T


table(list(SCORES), KEY_AT)

**Read it as:** if v1 is barely above chance on `pairs`, that is the gap this
whole module exists for. If "the product's ranking" beats v1, the photo framing is
doing work the critic is not.

## Step 4 — Train v2, and test it only on posts it never saw

**Folds by post.** The answers are split into 5 parts by *post*: every screen of a
post is on the same side. v2 is trained on 4 parts and scores the 5th, five times,
so every answer gets a score from a model that never saw that post. Splitting by
row instead would let the model see the same post's designs in training and test:
a flattering, wrong number.

**Two versions:**
- **lab only**: learns from the Taste Lab answers;
- **lab + synthetic**: also sees v1's broken-copy groups (weighted 0.3), so it keeps
  knowing that a broken design is worse. **This is the v2 the rule judges.** Lab-only
  is shown so you can see what the synthetic data adds or costs.

**Settings, fixed before looking:** smaller trees than v1 (7 leaves, at least 10
rows per leaf, a strong L2 penalty) and **100 trees**, because there are hundreds of
answers, not thousands of groups. Choosing the number of trees on these same
answers would flatter v2.

In [ ]:
PARAMS = {
    "objective": "lambdarank",
    "learning_rate": 0.05,
    "num_leaves": 7,
    "min_data_in_leaf": 10,
    "lambda_l2": 5.0,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "seed": 0,
    "deterministic": True,
    "force_row_wise": True,
    "num_threads": 1,
    "verbosity": -1,
}
ROUNDS = 100
SYNTH_WEIGHT = 0.3

started = time.time()
train_split, valid_split, _ = SPLITS
SYNTH = list(groups(ROOT, train_split, designs_per_copy=8, seed=0))
SYNTH_VALID = list(groups(ROOT, valid_split, designs_per_copy=8, seed=0))
took = time.time() - started
print(f"synthetic groups: {len(SYNTH)} to learn from, {len(SYNTH_VALID)} to check ({took:.0f} s)")


def base_wins(score, data):
    """Share of broken copies scored below their untouched design (v1's main mark)."""
    wins = n = 0.0
    for group in data:
        if group.kind == "broken":
            s = score(group.features)
            wins += (s[0] > s[1:]).sum() + 0.5 * (s[0] == s[1:]).sum()
            n += len(s) - 1
    return wins / n


def train(answers, synthetic=None):
    x, y, sizes = taste.stack(answers)
    w = taste.row_weights(answers)
    if synthetic:
        x = np.vstack([x, *[g.features for g in synthetic]])
        y = np.concatenate([y, *[g.labels for g in synthetic]])
        w = np.concatenate([w, np.full(sum(len(g.labels) for g in synthetic), SYNTH_WEIGHT)])
        sizes = sizes + [len(g.labels) for g in synthetic]
    data = lgb.Dataset(x, y, group=sizes, weight=w, feature_name=list(FEATURES))
    return lgb.train(PARAMS, data, num_boost_round=ROUNDS)


V2, LAB_ONLY = "v2 (lab + synthetic)", "v2 (lab only)"
FOLDS = taste.folds(TRAIN, k=5, seed=0)
SCORES[LAB_ONLY] = [None] * len(TRAIN)
SCORES[V2] = [None] * len(TRAIN)
forgetting = {LAB_ONLY: [], V2: []}
started = time.time()
for train_at, test_at in FOLDS:
    part = [TRAIN[i] for i in train_at]
    for name, synthetic in ((LAB_ONLY, None), (V2, SYNTH)):
        booster = train(part, synthetic)
        for i in test_at:
            SCORES[name][i] = booster.predict(TRAIN[i].features)
        forgetting[name].append(base_wins(booster.predict, SYNTH_VALID))
FORGETTING = {name: float(np.mean(values)) for name, values in forgetting.items()}
FORGETTING["critic v1"] = base_wins(V1.predict, SYNTH_VALID)
print(f"10 models trained in {time.time() - started:.0f} s")
print("synthetic base_wins (does it still know a broken design?):")
for name, value in FORGETTING.items():
    print(f"  {name}: {value:.3f}")

## Step 5 — Compare, on the answer key

Every row below was scored on answers its model never trained on (v1 and the
baselines never trained on any). Then the question that decides: **is v2 really
better than v1, or did it get lucky with which answers there were?** A *paired
bootstrap* draws the answers again with replacement, a thousand times, and scores
both models on the same draw each time. "wins" is the share of draws v2 came out
ahead.

In [ ]:
NAMES = ["chance", "the product's ranking", "hand-written rule", "critic v1", LAB_ONLY, V2]
display(table(NAMES, KEY_AT))

key_answers = [TRAIN[i] for i in KEY_AT]


def on_key(name):
    return [SCORES[name][i] for i in KEY_AT]


VS_PAIRS = taste.paired(on_key(V2), on_key("critic v1"), key_answers, key="pairs")
VS_TOP1 = taste.paired(on_key(V2), on_key("critic v1"), key_answers, key="top1")
pd.DataFrame({"pairs": VS_PAIRS, "top1": VS_TOP1}).T.round(3)

## Step 6 — The decision

The rule from the top, applied as written. Nothing here is adjusted after seeing
the numbers.

In [ ]:
wins, top1, broken = VS_PAIRS["wins"], VS_TOP1["difference"], FORGETTING[V2]
CHECKS = {
    f"learned from at least 150 answers (has {len(TRAIN)})": len(TRAIN) >= 150,
    f"orders pairs better than v1 in at least 95% of resamples ({wins:.0%})": wins >= 0.95,
    f"no worse than v1 at picking the best (top1 {top1:+.3f})": top1 >= 0,
    f"still knows a broken design (synthetic base_wins {broken:.3f} >= 0.95)": broken >= 0.95,
}
for check, passed in CHECKS.items():
    print("PASS" if passed else "FAIL", "-", check)
SHIP = all(CHECKS.values()) and not DRY_RUN
print()
if SHIP:
    print("v2 replaces v1.")
elif DRY_RUN:
    print("Dry run: nothing is saved.")
else:
    print("v1 stays. The report says which check failed; rate more, or recruit designers.")

## Step 7 — Train the final v2 on every answer, and switch it on (only if it passed)

The final model learns from all the answers with the same settings. The file it is
saved in carries proof rows: **fictional** synthetic designs and LightGBM's own
scores for them, so a test can check the numpy evaluator on every run. No client's
measurements go into the file.

Which measurements did v2 lean on? Gain is how much each one improved the trees.

In [ ]:
FINAL = train(TRAIN, SYNTH)
gain = pd.Series(FINAL.feature_importance("gain"), index=FEATURES)
share = (gain / gain.sum()).sort_values(ascending=False).head(10).round(3)
display(share.to_frame("share of gain"))

if SHIP:
    proof_x = np.vstack([group.features for group in SYNTH_VALID[:4]])
    save_model(
        FINAL,
        ROUNDS,
        MODELS / "critic-v2.json",
        proof_x=proof_x,
        meta={
            "trained": str(date.today()),
            "answers": len(TRAIN),
            "raters": len({one.rater for one in TRAIN}),
            "answer_key": KEY_NAME,
            "rounds": ROUNDS,
            "synthetic_weight": SYNTH_WEIGHT,
        },
    )
    print("saved", MODELS / "critic-v2.json")
    print("switched on:", write_active("critic-v2.json"))
else:
    print("not saved: the product keeps critic v1")

## Step 8 — The quality bar: would anyone send this at all?

**Labels.** Each design whose fate is known:
- *below the bar*: marked never send; not ticked as "would send"; or on a screen
  answered "none of these";
- *good enough*: the pick, and designs ticked "would send".

An older answer's unmarked designs are *unknown* (not chosen is not the same as not
good enough), so they are left out.

**The model** is a logistic regression on the same 35 measurements: small data, a
readable list of weights, and probabilities that are usually honest. It is checked
out of fold, by post, like v2.

**The threshold** comes from a stated cost: letting through a design a designer
would refuse costs **3** times as much as asking a person to look at one that was
fine. Above the threshold, the product puts other designs first, tries the repair
steps, and flags the post if nothing clears it. It never sends or deletes anything.

**The bar's rule:** at least 50 designs on each side, an out-of-fold AUC of at
least 0.70, and probabilities better than just knowing the base rate (Brier
score).

In [ ]:
BX, BAD, BW, SOURCE = taste.bar_rows(kept)
P = np.full(len(BAD), np.nan)
for _, test_at in taste.folds(kept, k=5, seed=0):
    held = np.isin(SOURCE, test_at)
    if held.any() and len(set(BAD[~held].tolist())) == 2:
        P[held] = quality.fit(BX[~held], BAD[~held], BW[~held]).p_bad(BX[held])
scored = np.isfinite(P)
p, bad = P[scored], BAD[scored]
AUC = quality.auc(p, bad)
BRIER, BASE = quality.brier(p, bad), quality.brier(np.full(len(bad), bad.mean()), bad)
THRESHOLD = quality.threshold_for(p, bad, miss_cost=quality.MISS_COST)
held_back = p > THRESHOLD
print(f"designs: {int(bad.sum())} below the bar, {int((1 - bad).sum())} good enough")
caught = (held_back & (bad == 1)).sum()
print(f"AUC {AUC:.3f} | Brier {BRIER:.3f} (base rate alone: {BASE:.3f})")
print(f"threshold {THRESHOLD:.3f}: {held_back.mean():.0%} of designs held for a look,")
print(f"  {caught / max(bad.sum(), 1):.0%} of the bad ones caught,")
print(f"  {caught / max(held_back.sum(), 1):.0%} of the held ones really bad")
pd.DataFrame(quality.calibration(p, bad, bins=5))

**Read the calibration table:** in each fifth of designs, sorted by the
predicted chance, "predicted" and "seen" should be close. If the bar says 0.7, about
70% of those designs should really have been refused.

In [ ]:
n_bad, n_good = int(bad.sum()), int((1 - bad).sum())
BAR_CHECKS = {
    f"at least 50 designs on each side ({n_bad} / {n_good})": min(n_bad, n_good) >= 50,
    f"ranks unseen designs well (AUC {AUC:.2f} >= 0.70)": AUC >= 0.70,
    f"better than the base rate alone (Brier {BRIER:.3f} < {BASE:.3f})": BRIER < BASE,
}
for check, passed in BAR_CHECKS.items():
    print("PASS" if passed else "FAIL", "-", check)
SHIP_BAR = all(BAR_CHECKS.values()) and not DRY_RUN

BAR = quality.fit(BX, BAD, BW)
BAR = replace(
    BAR,
    threshold=THRESHOLD,
    meta={"trained": str(date.today()), "designs": len(BAD), "auc_out_of_fold": round(AUC, 4)},
)
columns = ["measurement", "weight (+ pushes under the bar)"]
display(pd.DataFrame(BAR.weights()[:10], columns=columns))
if SHIP_BAR:
    quality.save(BAR)
    print("saved", quality.DEFAULT_BAR)
else:
    print("not saved: the product has no bar yet" + (" (dry run)" if DRY_RUN else ""))

## Step 9 — Write the report

Everything above, as numbers, in `ml/critic/report-v2.json` (a dry run writes to a
temporary folder instead). Fill `ml/critic/MODEL_CARD-v2.md` from it, using
`MODEL_CARD-v2.template.md`.

In [ ]:
def plain(value):
    if isinstance(value, dict):
        return {str(k): plain(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [plain(v) for v in value]
    if isinstance(value, (np.floating, float)):
        return None if not np.isfinite(value) else round(float(value), 4)
    if isinstance(value, (np.integer, np.bool_)):
        return value.item()
    return value


report = {
    "date": str(date.today()),
    "dry_run": DRY_RUN,
    "answers": {
        "kept": len(kept),
        "too_quick": len(too_fast),
        "training": len(TRAIN),
        "answer_key": len(KEY_AT),
    },
    "answer_key": KEY_NAME,
    "raters": rows,
    "ceiling": {"self_consistency": ceiling, "agreement": {"pairs": pairs, "rate": rate}},
    "marks_on_key": {
        name: taste.marks([SCORES[name][i] for i in KEY_AT], key_answers) for name in NAMES
    },
    "marks_all": {name: taste.marks(SCORES[name], TRAIN) for name in NAMES},
    "v2_vs_v1": {"pairs": VS_PAIRS, "top1": VS_TOP1},
    "synthetic_base_wins": FORGETTING,
    "decision": {"ship": SHIP, "checks": CHECKS},
    "settings": {
        "params": PARAMS,
        "rounds": ROUNDS,
        "synthetic_weight": SYNTH_WEIGHT,
        "min_ms": MIN_MS,
    },
    "bar": {
        "ship": SHIP_BAR,
        "checks": BAR_CHECKS,
        "auc": AUC,
        "brier": BRIER,
        "brier_base_rate": BASE,
        "threshold": THRESHOLD,
        "miss_cost": quality.MISS_COST,
        "calibration": quality.calibration(p, bad, bins=5),
        "weights": BAR.weights(),
    },
}
out = Path(tempfile.gettempdir()) / "report-v2-dry-run.json" if DRY_RUN else REPORT
out.write_text(json.dumps(plain(report), indent=2) + "\n", encoding="utf-8")
print("written", out)

## What to do now

**If v2 or the bar was switched on:** run `make check` (a test loads whatever the
switch names and checks its proof rows, and the bar loads), then commit
`apps/worker/src/stratum_worker/critic/model/` (critic-v2.json, ACTIVE, bar.json),
`ml/critic/report-v2.json` and the model card. To switch back: delete `ACTIVE` or
`bar.json`.

**If nothing passed:** commit `ml/critic/report-v2.json` alone. That is a result:
it says how far the data is from enough. More answers from designers is the fix.

**Five things to be able to say:**
1. v1 learned "a design beats a broken copy of itself"; v2 learns which of four real
   designs people would send, from Taste Lab answers.
2. Designers' answers are the answer key and count three times in training, so v2
   is not one person's taste.
3. Every number is out of fold by post; v2 replaces v1 only on a rule written
   before the results, with a paired bootstrap.
4. A ranker's best of a bad lot is still bad: the quality bar is a separate
   logistic model of "would not send", with a threshold set by a stated cost.
5. When nothing clears the bar, the product tries the other photo, the shortest
   headline, words-only designs and a wider search, then flags the post for a
   person. Nothing is ever sent by a model.